In [1]:
import polars as pl
import numpy as np
import json
import hashlib
from pathlib import Path
from scipy.sparse import coo_matrix, csr_matrix
import umap
from sklearn.manifold import trustworthiness
from itertools import product

In [2]:
umap_param_names = ["n_neighbors", "min_dist", "metric", "n_jobs", "random_state"]

In [3]:
# Make sure results folder exists
results_path = Path(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/output/nmf_umaps/umap_param_sweep/results.tsv"
)
if not results_path.exists():
    results_path.parent.mkdir(parents=True, exist_ok=True)
    header = umap_param_names + ["trustworthiness", "subset_n", "path"]
    with open(results_path, "w") as f:
            f.write("\t".join(header) + "\n")


In [4]:
def hash_params(params: dict, subset_n: int, replicate_num: int = 0) -> str:
    """
    Deterministic hash based on UMAP params + subset size.
    """
    s = json.dumps({"params": params, "subset_n": subset_n, "replicate_num": replicate_num}, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:16]  # 16-char hash

In [ ]:
def run_umap_and_log(
    full_mat: csr_matrix,
    pattern_meta: pl.DataFrame,
    params: dict,
    subset_n: int,
    results_tsv_path: Path,
    outdir: Path,
):
    """
    full_mat: csr_matrix (n_patterns x n_genes)
    pattern_meta: polars DataFrame with at least row_idx
    params: dict of UMAP constructor parameters
    subset_n: number of patterns to sample
    results_tsv_path: TSV for tracking runs
    outdir: where parquet outputs go
    """

    # -----------------------------------------
    # Subset selection
    # -----------------------------------------
    print("Sampling...")
    subset_meta = pattern_meta.sample(n=subset_n, shuffle=True).sort("row_idx")
    row_idx = subset_meta["row_idx"].to_numpy()
    subset_mat = full_mat[row_idx, :]

    # -----------------------------------------
    # Run UMAP
    # -----------------------------------------
    print(f"Running UMAP with params={params}")
    reducer = umap.UMAP(**params)
    embedding = reducer.fit_transform(subset_mat)

    # -----------------------------------------
    # Construct embedding df with metadata
    # -----------------------------------------
    embedding_df = pl.DataFrame(
        {
            "row_idx": row_idx,
            "UMAP1": embedding[:, 0],
            "UMAP2": embedding[:, 1],
        }
    )
    enriched = subset_meta.join(embedding_df, on="row_idx", how="left")
    print("Enriched embedding dataframe:")
    print(enriched)

    # -----------------------------------------
    # Trustworthiness
    # -----------------------------------------
    print("Calculating trustworthiness value...")
    trust = float(
        trustworthiness(subset_mat, embedding, n_neighbors=params.get("n_neighbors"))
    )
    print(f"trustworthiness={trust:.6f}")

    # -----------------------------------------
    # Save parquet
    # -----------------------------------------
    print("Saving to parquet...")
    outdir.mkdir(parents=True, exist_ok=True)
    run_hash = hash_params(params, subset_n)
    parquet_path = outdir / f"umap_sweep_{run_hash}.parquet"
    enriched.write_parquet(parquet_path)
    print(f"Saved to {parquet_path}")

    # -----------------------------------------
    # Append row to TSV
    # -----------------------------------------
    print("Appending row to TSV")
    with open(results_tsv_path, "a") as f:
        row = [str(params[p]) for p in umap_param_names] + [
            f"{trust:.6f}",
            str(subset_n),
            str(parquet_path),
        ]
        f.write("\t".join(row) + "\n")


In [6]:
ds = pl.scan_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/batched_nmf/all_patterns_partitioned"
)
df = (
    ds.filter(pl.col("loading") > 0)
    .with_columns([pl.struct(["run_id", "pattern"]).hash().alias("pattern_uid")])
    .with_columns(
        [
            pl.col("pattern_uid").rank(method="dense").cast(pl.UInt32).alias("row_idx")
            - 1,  # needs to be 0-indexed
            pl.col("gene")
            .cast(pl.Categorical)  # convert to categorical
            .cast(pl.UInt32)  # convert category codes -> integer indices
            .alias("col_idx"),
        ]
    )
    .drop("pattern_uid")
    .collect()
)
print(df)

shape: (211_093_586, 10)
┌────────┬─────────┬────────┬──────────┬───┬─────┬─────┬─────────┬─────────┐
│ run_id ┆ pattern ┆ gene   ┆ loading  ┆ … ┆ L1  ┆ k   ┆ row_idx ┆ col_idx │
│ ---    ┆ ---     ┆ ---    ┆ ---      ┆   ┆ --- ┆ --- ┆ ---     ┆ ---     │
│ i32    ┆ i32     ┆ str    ┆ f64      ┆   ┆ f64 ┆ i64 ┆ u32     ┆ u32     │
╞════════╪═════════╪════════╪══════════╪═══╪═════╪═════╪═════════╪═════════╡
│ 1      ┆ 1       ┆ FBXL18 ┆ 0.000058 ┆ … ┆ 0.1 ┆ 10  ┆ 33670   ┆ 0       │
│ 1      ┆ 2       ┆ FBXL18 ┆ 0.000039 ┆ … ┆ 0.1 ┆ 10  ┆ 33366   ┆ 0       │
│ 1      ┆ 3       ┆ FBXL18 ┆ 0.000075 ┆ … ┆ 0.1 ┆ 10  ┆ 41064   ┆ 0       │
│ 1      ┆ 4       ┆ FBXL18 ┆ 0.000029 ┆ … ┆ 0.1 ┆ 10  ┆ 3570    ┆ 0       │
│ 1      ┆ 5       ┆ FBXL18 ┆ 0.000063 ┆ … ┆ 0.1 ┆ 10  ┆ 21630   ┆ 0       │
│ …      ┆ …       ┆ …      ┆ …        ┆ … ┆ …   ┆ …   ┆ …       ┆ …       │
│ 749    ┆ 53      ┆ DHRSX  ┆ 0.000061 ┆ … ┆ 0.7 ┆ 90  ┆ 31596   ┆ 743     │
│ 749    ┆ 59      ┆ DHRSX  ┆ 0.000046 ┆ … ┆ 0.7 ┆ 

In [7]:
pattern_meta = (
    df.select(pl.exclude("gene", "loading", "col_idx")).unique().sort("row_idx")
)
gene_meta = df.select("gene", "col_idx").unique().sort("col_idx")
print(pattern_meta)
print(gene_meta)

shape: (41_250, 7)
┌────────┬─────────┬────────┬───────────┬─────┬─────┬─────────┐
│ run_id ┆ pattern ┆ seed   ┆ tol       ┆ L1  ┆ k   ┆ row_idx │
│ ---    ┆ ---     ┆ ---    ┆ ---       ┆ --- ┆ --- ┆ ---     │
│ i32    ┆ i32     ┆ i32    ┆ f64       ┆ f64 ┆ i64 ┆ u32     │
╞════════╪═════════╪════════╪═══════════╪═════╪═════╪═════════╡
│ 649    ┆ 64      ┆ 2025   ┆ 0.00001   ┆ 0.7 ┆ 90  ┆ 0       │
│ 452    ┆ 9       ┆ 42     ┆ 0.00001   ┆ 0.5 ┆ 20  ┆ 1       │
│ 640    ┆ 42      ┆ 120301 ┆ 0.00001   ┆ 0.7 ┆ 100 ┆ 2       │
│ 718    ┆ 2       ┆ 1029   ┆ 0.0000001 ┆ 0.7 ┆ 80  ┆ 3       │
│ 715    ┆ 50      ┆ 1029   ┆ 0.0000001 ┆ 0.7 ┆ 50  ┆ 4       │
│ …      ┆ …       ┆ …      ┆ …         ┆ …   ┆ …   ┆ …       │
│ 139    ┆ 25      ┆ 120301 ┆ 0.0000001 ┆ 0.1 ┆ 90  ┆ 41245   │
│ 269    ┆ 36      ┆ 1029   ┆ 0.0000001 ┆ 0.0 ┆ 90  ┆ 41246   │
│ 420    ┆ 49      ┆ 1029   ┆ 0.0000001 ┆ 0.2 ┆ 100 ┆ 41247   │
│ 388    ┆ 63      ┆ 120301 ┆ 0.000001  ┆ 0.2 ┆ 80  ┆ 41248   │
│ 275    ┆ 23      ┆ 

In [8]:
rows = df["row_idx"].to_numpy()
cols = df["col_idx"].to_numpy()
vals = df["loading"].to_numpy()

n_rows = int(rows.max()) + 1
n_cols = int(cols.max()) + 1

full_mat = coo_matrix((vals, (rows, cols)), shape=(n_rows, n_cols)).tocsr()
print(full_mat)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 211065934 stored elements and shape (41250, 34978)>
  Coords	Values
  (0, 292)	0.03527543237641608
  (0, 850)	0.008209300138483299
  (0, 1065)	0.06340401462705621
  (0, 1297)	0.16741380845738932
  (0, 1459)	0.2180172246969571
  (0, 1508)	0.004290548294765327
  (0, 1852)	0.0025053247631701917
  (0, 2586)	0.07718955795771366
  (0, 2960)	0.01694557216350994
  (0, 5531)	0.02267467733575984
  (0, 5646)	0.004892986811502506
  (0, 6603)	0.005975823513134125
  (0, 7092)	0.06384950266400204
  (0, 7182)	0.05223535442623298
  (0, 7822)	0.10936403750066433
  (0, 8783)	0.008527800388332915
  (0, 10614)	0.007353307893484654
  (0, 10934)	0.04414961625963279
  (0, 11692)	0.013397035261785128
  (0, 12260)	0.07255452814778364
  (0, 18821)	0.0012402015414140991
  (0, 18887)	0.0005343447808100523
  (1, 1)	3.621385899235499e-05
  (1, 3)	8.287601284377682e-05
  (1, 9)	0.00041205687579947195
  :	:
  (41249, 34912)	1.6701886338910847e-06
  (41249, 

In [9]:
param_grid = {
    "n_neighbors": [5, 10, 15, 20, 50, 100, 200, 500, 1_000],
    "min_dist": [0.0, 0.01, 0.1, 0.25, 0.5],
    "metric": ["cosine", "euclidean"],
    "n_jobs": [64],          
    "random_state": [None],  
}

def iterate_param_grid(param_grid):
    """
    Convert dict-of-lists into dicts for each combination.
    """
    keys = list(param_grid.keys())
    value_lists = [param_grid[k] for k in keys]

    for values in product(*value_lists):
        yield dict(zip(keys, values))

In [ ]:
subset_n = 5_000
outdir = Path("/zata/zippy/kresgeb/nmf_stuff/dlPFC/output/nmf_umaps/umap_param_sweep/enriched_embeddings")

for params in iterate_param_grid(param_grid):
    run_umap_and_log(
            full_mat=full_mat,
            pattern_meta=pattern_meta,
            params=params,
            subset_n=subset_n,
            results_tsv_path=results_path,
            outdir=outdir,
        )

Sampling...
 Running UMAP with params={'n_neighbors': 5, 'min_dist': 0.0, 'metric': 'cosine', 'n_jobs': 64, 'random_state': None}
Enriched embedding dataframe:
shape: (5_000, 9)
┌────────┬─────────┬────────┬───────────┬───┬─────┬─────────┬───────────┬───────────┐
│ run_id ┆ pattern ┆ seed   ┆ tol       ┆ … ┆ k   ┆ row_idx ┆ UMAP1     ┆ UMAP2     │
│ ---    ┆ ---     ┆ ---    ┆ ---       ┆   ┆ --- ┆ ---     ┆ ---       ┆ ---       │
│ i32    ┆ i32     ┆ i32    ┆ f64       ┆   ┆ i64 ┆ u32     ┆ f32       ┆ f32       │
╞════════╪═════════╪════════╪═══════════╪═══╪═════╪═════════╪═══════════╪═══════════╡
│ 624    ┆ 15      ┆ 31415  ┆ 0.00001   ┆ … ┆ 40  ┆ 22      ┆ -5.14318  ┆ 1.645687  │
│ 300    ┆ 47      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 100 ┆ 31      ┆ -8.956904 ┆ 2.000146  │
│ 335    ┆ 37      ┆ 120301 ┆ 0.00001   ┆ … ┆ 50  ┆ 36      ┆ -8.443942 ┆ -1.429973 │
│ 298    ┆ 31      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 80  ┆ 43      ┆ 5.414066  ┆ 0.385859  │
│ 79     ┆ 60      ┆ 31415  ┆ 0.000001  ┆ … ┆ 90

/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Enriched embedding dataframe:
shape: (5_000, 9)
┌────────┬─────────┬────────┬───────────┬───┬─────┬─────────┬────────────┬────────────┐
│ run_id ┆ pattern ┆ seed   ┆ tol       ┆ … ┆ k   ┆ row_idx ┆ UMAP1      ┆ UMAP2      │
│ ---    ┆ ---     ┆ ---    ┆ ---       ┆   ┆ --- ┆ ---     ┆ ---        ┆ ---        │
│ i32    ┆ i32     ┆ i32    ┆ f64       ┆   ┆ i64 ┆ u32     ┆ f32        ┆ f32        │
╞════════╪═════════╪════════╪═══════════╪═══╪═════╪═════════╪════════════╪════════════╡
│ 452    ┆ 9       ┆ 42     ┆ 0.00001   ┆ … ┆ 20  ┆ 1       ┆ -7.218578  ┆ -5.798798  │
│ 147    ┆ 61      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 70  ┆ 13      ┆ 16.637526  ┆ 7.491128   │
│ 466    ┆ 51      ┆ 1029   ┆ 0.00001   ┆ … ┆ 60  ┆ 18      ┆ 20.039967  ┆ 16.022987  │
│ 299    ┆ 80      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 90  ┆ 29      ┆ -3.180821  ┆ -12.857433 │
│ 300    ┆ 47      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 100 ┆ 31      ┆ -10.109785 ┆ -7.81708   │
│ …      ┆ …       ┆ …      ┆ …         ┆ … ┆ …   ┆ …       ┆ …         